# 06_01 One neuron: what can it learn, and what can it never learn?

A neuron is a weighted sum and a decision. This notebook builds one in NumPy, lets it learn two logical
functions in a handful of passes, and then asks it to learn a third, which it will fail at forever. By the
end you will know exactly why, and it is the reason every network since has hidden layers.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-06-a-neuron-from-scratch", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'sentence_transformers': 'sentence-transformers',
           'torch': 'torch',
           'sklearn': 'scikit-learn',
           'pandas': 'pandas',
           'numpy': 'numpy',
           'matplotlib': 'matplotlib'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
from nlpcheck import ask, guess, reveal, check_06_01

# The four possible inputs of a two-input logical function, one per row.
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
TARGETS = {"AND": [0, 0, 0, 1], "OR": [0, 1, 1, 1], "XOR": [0, 1, 1, 0]}
print(X)

## 1. Recall

From earlier labs. Answer from memory, then run the cell.

**r1.** In Lab 03, logistic regression turned a weighted sum into a probability with which function?
(a) ReLU, (b) sigmoid, (c) softmax

**r2.** What is a sentence embedding? (a) a count of each word, (b) a list of numbers that places a
sentence by its meaning, (c) a tokenizer's vocabulary

In [ ]:
ask("r1", "")   # put your letter between the quotes
ask("r2", "")

## 2. A neuron by hand

McCulloch and Pitts' neuron (1943) has three parts: a **weight** for each input, an **adder** that sums the
weighted inputs plus a **bias**, and an **activation** that decides whether the neuron fires. The simplest
activation is a step: fire (1) if the sum is above zero, otherwise stay quiet (0).

Here is a neuron with weights 1 and 1 and a bias of -1.5. Predict which logical function it computes:
`and`, `or` or `xor`.

In [ ]:
def neuron(x, w, b):
    total = x @ w + b              # the adder: a weighted sum plus the bias
    return (total > 0).astype(int) # the activation: a step at zero

guess("which_function", None)   # "and", "or" or "xor" 

In [ ]:
out = neuron(X, np.array([1, 1]), -1.5)
for x, y in zip(X, out):
    print(x, "->", y)
reveal("which_function", "and" if out.tolist() == TARGETS["AND"] else "or" if out.tolist() == TARGETS["OR"] else "other")
print("with bias -0.5:", neuron(X, np.array([1, 1]), -0.5).tolist())

AND: the sum 1 + 1 - 1.5 is positive only when both inputs are 1. Move the bias to -0.5 and one input is
enough, so the same neuron computes OR. The bias is a threshold in disguise: it says how much evidence the
neuron needs before it fires.

## 3. Letting it learn: the perceptron

Nobody should have to choose the weights by hand. Rosenblatt's **perceptron** (1958) learns them from
examples with one rule. Show it an example; if its output `y` is wrong, nudge every weight in the direction
that would have made it right:

`w = w + lr * (target - y) * x`

`lr` is the **learning rate**, how big a nudge. If the answer was right, `target - y` is 0 and nothing
changes. The bias is handled as one more weight on an input that is always -1, so the same rule learns it.

**Your turn.** Finish `perceptron_step`: compute `y` from the weights, apply the rule, and return the new
weights and `y`. As shipped it never changes the weights, so it can never learn.

In [ ]:
def perceptron_step(w, x, target, lr):
    y = 1 if x @ w > 0 else 0
    # YOUR CODE HERE: apply the perceptron rule to w
    return w, y

def train_perceptron(targets, epochs=20, lr=0.25, seed=0):
    rng = np.random.default_rng(seed)
    w = rng.uniform(-0.05, 0.05, 3)              # small random starting weights: bias, w1, w2
    Xb = np.hstack([-np.ones((4, 1)), X])        # the constant -1 input that carries the bias
    errors = []
    for epoch in range(epochs):                  # one epoch is one pass over all four examples
        mistakes = 0
        for x, t in zip(Xb, targets):
            w, y = perceptron_step(w, x, t, lr)
            mistakes += int(y != t)
        errors.append(mistakes)
    accuracy = float(np.mean([(1 if x @ w > 0 else 0) == t for x, t in zip(Xb, targets)]))
    return w, errors, accuracy

for name in ("AND", "OR"):
    w, errors, acc = train_perceptron(TARGETS[name])
    print(f"{name}: mistakes per epoch {errors[:8]}  accuracy {acc}")

When `perceptron_step` is right, AND makes 1, 3, 3, 2, 1 mistakes in its first five epochs and none from the
sixth on, and OR is perfect from the fourth. The mistakes are the learning: every one moved the weights.

## 4. XOR

**XOR** (exclusive or) is 1 when exactly one input is 1. It looks no harder than AND or OR. Predict: how
many epochs will the perceptron need before it makes no mistakes on XOR? (a number; if you think never,
write `never`)

In [ ]:
guess("xor_epochs", None)

In [ ]:
w, errors, acc = train_perceptron(TARGETS["XOR"], epochs=1000)
print("mistakes in the first 10 epochs:", errors[:10])
print("mistakes in the last 10 epochs: ", errors[-10:])
print("epochs with no mistakes:", sum(e == 0 for e in errors), "of 1000;  accuracy", acc)
reveal("xor_epochs", "never" if 0 not in errors else errors.index(0) + 1)

Never. After the first few epochs it gets **all four** examples wrong on every pass, for a thousand passes.
More epochs will not help, and neither will a smaller learning rate. The picture shows why.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, name in zip(axes, ("AND", "OR", "XOR")):
    t = np.array(TARGETS[name])
    ax.scatter(X[t == 0, 0], X[t == 0, 1], s=200, marker="o", label="0")
    ax.scatter(X[t == 1, 0], X[t == 1, 1], s=200, marker="s", label="1")
    if name != "XOR":
        w, _, _ = train_perceptron(t)
        xs = np.linspace(-0.5, 1.5, 10)
        ax.plot(xs, (w[0] - w[1] * xs) / w[2], "k--")   # the line where the weighted sum is zero
    ax.set_title(name); ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5); ax.legend()
plt.show()

A perceptron can only draw **one straight line** (in more dimensions, a flat plane) and fire on one side of
it. AND and OR each have a line that separates the 1s from the 0s. XOR does not: the two 1s sit on opposite
corners, and any line that puts both on one side takes a 0 with them. The classes are **not linearly
separable**, so the rule keeps chasing its own last mistake round the four corners. The fix is not a better
rule; it is more than one neuron, which is the next notebook.

## 5. Save and check

In [ ]:
os.makedirs("out", exist_ok=True)
result = {}
for name, t in TARGETS.items():
    w, errors, acc = train_perceptron(t)
    result[name] = {"weights": [float(v) for v in w], "errors": errors, "accuracy": acc}
json.dump(result, open("out/06_01_perceptron.json", "w"), indent=1)
check_06_01()

## 6. Exit ticket

The book lists four steps of a perceptron: (1) initialise the weights randomly, (2) go to the next batch of
data, (3) if the prediction does not match the output, change the weights, (4) for a sample input, compute
an output.

**x1.** Which is the correct order? (a) 1, 2, 3, 4, (b) 4, 3, 2, 1, (c) 3, 1, 2, 4, (d) 1, 4, 3, 2

In [ ]:
ask("x1", "")

Explain it back: why does training a perceptron longer never fix XOR? One or two sentences.

*Your explanation:* 